# ЛР-2: Parquet benchmark

Среда: **Google Colab**.

Полное методическое описание, критерии оценивания и `TEACH CARD` находятся в одноимённом `.md` файле комплекта.

Сквозной пайплайн курса:

$$
\text{Sense} \rightarrow \text{Collect} \rightarrow \text{Stream} \rightarrow
\text{Store} \rightarrow \text{Process} \rightarrow \text{Learn} \rightarrow \text{Teach}
$$


In [ ]:
!pip -q install "pyspark==4.2.0" pandas pyarrow

from pyspark.sql import SparkSession, functions as F
from pathlib import Path
import shutil
import time
import statistics

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab02_ParquetBenchmark")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

ROOT = Path("/content/lab02_storage")
CSV_DIR = ROOT / "csv"
PARQUET_DIR = ROOT / "parquet"
PARTITIONED_DIR = ROOT / "parquet_partitioned"

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)

N = 350_000

df = (
    spark.range(N)
    .withColumn("robot_id", F.concat(F.lit("robot-"), (F.col("id") % 12).cast("string")))
    .withColumn("sensor", F.when(F.col("id") % 2 == 0, F.lit("imu")).otherwise(F.lit("motor")))
    .withColumn("ts_ms", (F.lit(1_720_000_000_000) + F.col("id") * 100).cast("long"))
    .withColumn("temperature", F.lit(40.0) + (F.col("id") % 150) * 0.08)
    .withColumn("vibration", F.abs(F.sin(F.col("id") / 15.0)) + (F.col("id") % 11) * 0.002)
    .withColumn("payload", F.sha2(F.col("id").cast("string"), 256))
    .drop("id")
)

df.write.mode("overwrite").option("header", True).csv(str(CSV_DIR))
df.write.mode("overwrite").option("compression", "snappy").parquet(str(PARQUET_DIR))
df.write.mode("overwrite").partitionBy("robot_id").parquet(str(PARTITIONED_DIR))

def dir_size(path: Path) -> int:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

sizes = {
    "CSV": dir_size(CSV_DIR),
    "Parquet": dir_size(PARQUET_DIR),
    "Partitioned Parquet": dir_size(PARTITIONED_DIR),
}

for name, size in sizes.items():
    print(f"{name:22s}: {size / 1024 / 1024:.2f} MiB")

compression_ratio = sizes["CSV"] / sizes["Parquet"]
print(f"CSV/Parquet size ratio: {compression_ratio:.2f}x")

def benchmark_read(fmt, path, predicate=None, repeats=3):
    times = []
    counts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        if fmt == "csv":
            x = spark.read.option("header", True).option("inferSchema", True).csv(str(path))
        else:
            x = spark.read.parquet(str(path))
        if predicate is not None:
            x = x.filter(predicate)
        counts.append(x.count())
        times.append(time.perf_counter() - t0)
    return statistics.median(times), counts[-1]

csv_t, csv_n = benchmark_read("csv", CSV_DIR, F.col("robot_id") == "robot-7")
pq_t, pq_n = benchmark_read("parquet", PARQUET_DIR, F.col("robot_id") == "robot-7")
part_t, part_n = benchmark_read("parquet", PARTITIONED_DIR, F.col("robot_id") == "robot-7")

print(f"CSV                : {csv_t:.3f}s, rows={csv_n}")
print(f"Parquet            : {pq_t:.3f}s, rows={pq_n}")
print(f"Partitioned Parquet: {part_t:.3f}s, rows={part_n}")

print("Speedup CSV -> Parquet:", round(csv_t / pq_t, 2), "x")
print("Speedup CSV -> partitioned:", round(csv_t / part_t, 2), "x")

# Проверяем predicate/partition pruning в плане.
(
    spark.read.parquet(str(PARTITIONED_DIR))
    .filter(F.col("robot_id") == "robot-7")
    .select("robot_id", "temperature")
    .explain(mode="formatted")
)

assert csv_n == pq_n == part_n
assert sizes["Parquet"] < sizes["CSV"]


## TEACH CARD

После выполнения кода заполните `TEACH CARD` из `.md`-файла лабораторной работы и приложите его к отчёту.
